# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This lane is best described as ranking with a classification-style target. The end product is a ranked queue of pages to review first, so the task is ranking/scoring. At the same time, the starter label is a binary decline signal, which means the model is trained like a classifier and then used to rank pages by probability.

That is the right frame for Lane 2: the label is per-page decline risk, and the output is a score that orders pages by refresh priority. So the task is ranking, but it uses a supervised classification proxy to learn the ranking.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

from pathlib import Path
raw = Path("data/raw/content_refresh_anonymized.csv")
if not raw.exists():
    raw = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(raw)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Data rows: {len(df):,}")
print(f"Declining pages: {int(df['is_declining_label'].sum()):,}")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")
print(f"Unique pages: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique():,}")

Data rows: 30,000
Declining pages: 16,262
Declining rate: 54.2%
Unique pages: 30,000
Unique clients: 32


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: whether a page is currently declining in impressions. The label is `is_declining_label`, which comes from `trend_direction == "down"` in the starter data.

That label is a proxy for refresh opportunity. It is derived from the current performance window, so it is not the perfect future impact label. It is still observed from the data, not a hidden product decision, and it gives us a consistent way to train a model on the starter slice.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The target is observed from the current window, not inferred from a product score.
print("Label source: is_declining_label from trend_direction == 'down'")
print(f"Distinct trend directions: {df['trend_direction'].nunique()}")
print(df['trend_direction'].value_counts(normalize=True).round(3))

Label source: is_declining_label from trend_direction == 'down'
Distinct trend directions: 5
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: precision@50. This fits the limited refresh slot decision because it measures how many of the top 50 ranked pages are actually labeled as declining. A good result is meaningfully higher than the baseline rule's precision@50.

I will also track recall@50 as a secondary metric, because it shows how many of the relevant declining pages the top 50 capture. Precision at the top of the queue is the main judge, with recall@50 as extra evidence of ranking quality.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page. That matches the decision we are supporting: choose which pages to refresh first.

The starter CSV is already page-level, so the analysis is naturally aligned with the action. I will make sure the row grain stays consistent and that I do not accidentally treat page-level features as if they were time-series instances.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(df.columns.tolist())
print(df.head(1).T)

Rows: 30,000
Columns: 45
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']
                                           0
content_id              content_304f48230142
client_id                  client_f369cb89fc
search_volume                           10.0
co

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The patterns that make a page decline are not simple thresholds. A page could be declining because it is old, because its impressions are dropping, because its engagement is low, or because its recent demand is weak. Those signals also interact: a page with high impressions and fast decline is different from one with low impressions and slow decline.

A fixed rule can only capture a few of those cases. ML can combine signals across traffic, trend, age, engagement, and content metadata to score pages in a richer way. That is why ranking by a learned probability is more promising than a single hand rule for this lane.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.